## 1. Instalasi Dependensi

In [1]:

!pip install -U pip

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters

!pip install \
langchain==0.1.20 \
langchain-core==0.1.52 \
langchain-community==0.0.38 \
langchain-text-splitters==0.0.2

!pip install \
faiss-gpu \
rank_bm25 \
sentence-transformers \
pypdf \
gradio \
duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-1htj5sw0/unsloth_c1104357d2ac4ab7a21ff21a423a9807
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-1htj5sw0/unsloth_c1104357d2ac4ab7a21ff21a423a9807
  Resolved https://github.com/unslothai/unsloth.git to commit e1ae4756d9afe7fc5ba34e939fe5ba75dfb6e94e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 74.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 64.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 104.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 103.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 3/5 [faiss-cpu]ERROR: Operation cancelled by user
^C


In [2]:
  pip install duckduckgo-search

  Using cached duckduckgo_search-8.1.1-py3-none-any.whl.metadata (16 kB)
Using cached duckduckgo_search-8.1.1-py3-none-any.whl (18 kB)


## 2. Import Library

In [3]:
import re
import math
import numpy as np
import torch

from unsloth import FastLanguageModel
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.embeddings import Embeddings

from sentence_transformers import SentenceTransformer, CrossEncoder
from duckduckgo_search import DDGS
import gradio as gr

print("CUDA tersedia:", torch.cuda.is_available())


CUDA tersedia: True


## 3. Menyiapkan 4 Dokumen PDF

In [4]:

PDF_FILENAMES = [
    "PP Nomor 5 Tahun 2021.pdf",
    "PP Nomor 35 Tahun 2021.pdf",
    "PP Nomor 51 Tahun 2023.pdf",
    "UU Nomor 6 Tahun 2023.pdf",
]

import os
for fn in PDF_FILENAMES:
    assert os.path.exists(fn), f"File belum ada di working directory: {fn}"
print("Semua 4 file PDF ditemukan.")

Semua 4 file PDF ditemukan.


## 4. Memuat PDF + Metadata Enrichment

In [28]:
def enrich_metadata(documents, filename):
    match = re.match(r"(UU|PP) Nomor (\d+) Tahun (\d+)", filename)
    jenis = match.group(1) if match else "UNKNOWN"
    nomor = match.group(2) if match else "?"
    tahun = match.group(3) if match else "?"
    label = f"{jenis} No. {nomor}/{tahun}"
    for doc in documents:
        doc.metadata["jenis_peraturan"] = jenis
        doc.metadata["nomor_peraturan"] = label
        doc.metadata["tahun"] = tahun
        doc.metadata["filename"] = filename
    return documents

all_page_documents = []
for fn in PDF_FILENAMES:
    loader = PyPDFLoader(fn)
    pages = loader.load()
    pages = enrich_metadata(pages, fn)
    all_page_documents.extend(pages)
    print(f"{fn:35s} -> {len(pages)} halaman dimuat")

print(f"\nTotal halaman (Document) dari 4 dokumen: {len(all_page_documents)}")
print("\nContoh metadata halaman pertama:", all_page_documents[0].metadata)


PP Nomor 5 Tahun 2021.pdf           -> 739 halaman dimuat
PP Nomor 35 Tahun 2021.pdf          -> 56 halaman dimuat
PP Nomor 51 Tahun 2023.pdf          -> 27 halaman dimuat
UU Nomor 6 Tahun 2023.pdf           -> 1127 halaman dimuat

Total halaman (Document) dari 4 dokumen: 1949

Contoh metadata halaman pertama: {'source': 'PP Nomor 5 Tahun 2021.pdf', 'page': 0, 'jenis_peraturan': 'PP', 'nomor_peraturan': 'PP No. 5/2021', 'tahun': '2021', 'filename': 'PP Nomor 5 Tahun 2021.pdf'}


## 5. Text Splitting

In [29]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

child_chunks = child_splitter.split_documents(all_page_documents)
print(f"Total child chunks: {len(child_chunks)}")
print("\nContoh chunk pertama:")
print(child_chunks[0].page_content[:300])
print("Metadata:", child_chunks[0].metadata)


Total child chunks: 3234

Contoh chunk pertama:
LEMBARAN NEGARA 
REPUBLIK INDONESIA 
No.15, 2021 ADMINISTRASI. Perizinan Berusaha Berbasis 
Risiko. Penyelenggaraan. (Penjelasan dalam 
Tambahan Lembaran Negara Republik 
Indonesia Nomor 6617) 
 
PERATURAN PEMERINTAH REPUBLIK INDONESIA 
NOMOR 5 TAHUN 2021 
TENTANG 
PENYELENGGARAAN PERIZINAN BERUSAHA
Metadata: {'source': 'PP Nomor 5 Tahun 2021.pdf', 'page': 0, 'jenis_peraturan': 'PP', 'nomor_peraturan': 'PP No. 5/2021', 'tahun': '2021', 'filename': 'PP Nomor 5 Tahun 2021.pdf'}


## 6. Embedding Model + Vector DB

In [7]:
class E5Embeddings(Embeddings):
    def __init__(self, model_name="intfloat/multilingual-e5-base", device=None):
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts):
        prefixed = [f"passage: {t}" for t in texts]
        vectors = self.model.encode(
            prefixed, batch_size=32, normalize_embeddings=True, show_progress_bar=True
        )
        return vectors.tolist()

    def embed_query(self, text):
        vector = self.model.encode(f"query: {text}", normalize_embeddings=True)
        return vector.tolist()


embeddings = E5Embeddings()


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [30]:
faiss_vectorstore = FAISS.from_documents(child_chunks, embeddings)
faiss_vectorstore.save_local("faiss_index_legal_docs")
print("FAISS vector store berhasil dibangun dan disimpan ke ./faiss_index_legal_docs")

Batches:   0%|          | 0/102 [00:00<?, ?it/s]

FAISS vector store berhasil dibangun dan disimpan ke ./faiss_index_legal_docs


In [11]:
sample_results = faiss_vectorstore.similarity_search("upah lembur staf admin", k=3)
for i, doc in enumerate(sample_results, 1):
    print(f"[{i}] {doc.metadata['nomor_peraturan']} (hal. {doc.metadata.get('page')})")
    print(doc.page_content[:200].replace(chr(10), ' '))
    print()

[1] UNKNOWN No. ?/? (hal. 18)
PRESIDEN REPUELIK INDONESIA -L9- a. untuk ja- kerja lembur pertama sebesar 1,5 (satu koma lima) kali Upah sejam; dan b. untuk setiap ja- kerja lembur berikutnya, sebesar 2 (dua) kali Upah sejam. (2) P

[2] UNKNOWN No. ?/? (hal. 16)
(41 Pengaturan golongan jabatan tertentu diatur dalam Perjanjian Kerja, Peraturan Perusahaan, atau Perjanjian Kerja Bersama. (5) Apabila golongan jabatan tertentu tidak diatur dalam Perjanjian Kerja, 

[3] UNKNOWN No. ?/? (hal. 19)
PRESIDEN REPUBLIK INDONESIA -20- b. jam kesembilan, dibayar 3 (tiga) kali Upah sejam; dan c. jam kesepuluh, jam kesebelas, dan jam kedua belas, dibayar 4 (empat) kali Upah sejam. Pasal 32 (1) Perhitun



## 7. Memuat Model Hasil Fine-tuning + Prompt Template


In [12]:
FT_MODEL_REPO = "snssamuel/qwen2.5-3b-legal-chatbot-id"

llm_model, llm_tokenizer = FastLanguageModel.from_pretrained(
    model_name=FT_MODEL_REPO,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(llm_model)


==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

config.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.21k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen

In [33]:
PROMPT_TEMPLATE = (
    "Anda adalah asisten hukum internal yang menjawab pertanyaan tim legal berdasarkan "
    "dokumen peraturan resmi berikut. Jawab HANYA berdasarkan konteks yang diberikan. "
    "Jika konteks tidak cukup untuk menjawab, katakan dengan tegas bahwa informasinya "
    "tidak ditemukan di dokumen.\n\n"
    "Konteks:\n{context}\n\n"
    "Pertanyaan: {question}\n\n"
    "Jawaban:"
)

def generate_answer(prompt_text, max_new_tokens=400):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = llm_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = llm_model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens, use_cache=True
    )
    new_tokens = outputs[0][inputs.shape[-1]:]
    answer = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return answer.strip()

## 8. Pipeline RAG Dasar + Interface

In [34]:
def simple_rag_query(question, k=3):
    docs = faiss_vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(
        f"[Sumber: {d.metadata['nomor_peraturan']}, hal. {d.metadata.get('page')}]\n{d.page_content}"
        for d in docs
    )
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    return generate_answer(prompt)


In [15]:
basic_demo = gr.Interface(
    fn=simple_rag_query,
    inputs=gr.Textbox(label="Pertanyaan"),
    outputs=gr.Textbox(label="Jawaban"),
    title="Chatbot Tim Legal - RAG Basic",
)
basic_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ce837d8a2d51ed24e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 9. Ensemble Retriever: BM25 + Semantik

In [31]:
bm25_retriever = BM25Retriever.from_documents(child_chunks)
bm25_retriever.k = 5

vector_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 5})

BM25_WEIGHT = 0.4
SEMANTIC_WEIGHT = 0.6

from langchain.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[BM25_WEIGHT, SEMANTIC_WEIGHT],
)

ensemble_results = ensemble_retriever.invoke("upah lembur staf admin")
print(f"Jumlah dokumen hasil ensemble retriever: {len(ensemble_results)}")
for i, doc in enumerate(ensemble_results, 1):
    print(f"[{i}] {doc.metadata['nomor_peraturan']} (hal. {doc.metadata.get('page')})")


Jumlah dokumen hasil ensemble retriever: 9
[1] PP No. 35/2021 (hal. 18)
[2] PP No. 35/2021 (hal. 16)
[3] PP No. 35/2021 (hal. 19)
[4] PP No. 35/2021 (hal. 18)
[5] PP No. 35/2021 (hal. 2)
[6] PP No. 35/2021 (hal. 17)
[7] UU No. 6/2023 (hal. 977)
[8] UU No. 6/2023 (hal. 558)
[9] UU No. 6/2023 (hal. 1065)


## 10. Metadata Filtering + Sitasi Jawaban

In [17]:
def search_with_filter(query, jenis_peraturan=None, nomor_peraturan=None, k=5):
    filter_dict = {}
    if jenis_peraturan:
        filter_dict["jenis_peraturan"] = jenis_peraturan
    if nomor_peraturan:
        filter_dict["nomor_peraturan"] = nomor_peraturan
    return faiss_vectorstore.similarity_search(query, k=k, filter=filter_dict or None)

filtered_results = search_with_filter(
    "upah lembur", nomor_peraturan="PP No. 35/2021", k=3
)
for doc in filtered_results:
    print(doc.metadata["nomor_peraturan"], "- hal.", doc.metadata.get("page"))


In [18]:
def format_context_with_citations(docs):
    context_parts = []
    citations = []
    for i, doc in enumerate(docs, 1):
        label = f"{doc.metadata.get('nomor_peraturan', '?')}, hal. {doc.metadata.get('page', '?')}"
        context_parts.append(f"[{i}] (Sumber: {label})\n{doc.page_content}")
        citations.append(f"[{i}] {label}")
    context = "\n\n".join(context_parts)
    return context, citations


## 11. Parent-Child Retriever

In [32]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
pc_child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

pc_vectorstore = FAISS.from_texts(["placeholder"], embeddings)
pc_docstore = InMemoryStore()

parent_child_retriever = ParentDocumentRetriever(
    vectorstore=pc_vectorstore,
    docstore=pc_docstore,
    child_splitter=pc_child_splitter,
    parent_splitter=parent_splitter,
)

parent_child_retriever.add_documents(all_page_documents)
print("Parent-Child retriever siap. Jumlah parent docs:", len(list(pc_docstore.yield_keys())))


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Parent-Child retriever siap. Jumlah parent docs: 1962


In [20]:
pc_results = parent_child_retriever.invoke("upah lembur staf admin")
print(f"Jumlah parent chunk dikembalikan: {len(pc_results)}")
for doc in pc_results[:2]:
    print("Panjang parent chunk:", len(doc.page_content), "karakter")
    print(doc.page_content[:200].replace(chr(10), " "))
    print()


Jumlah parent chunk dikembalikan: 3
Panjang parent chunk: 1556 karakter
PRESIDEN REPUELIK INDONESIA -L9- a. untuk ja- kerja lembur pertama sebesar 1,5 (satu koma lima) kali Upah sejam; dan b. untuk setiap ja- kerja lembur berikutnya, sebesar 2 (dua) kali Upah sejam. (2) P

Panjang parent chunk: 1624 karakter
PRESIDEN REPUBLIK INDONESIA -20- b. jam kesembilan, dibayar 3 (tiga) kali Upah sejam; dan c. jam kesepuluh, jam kesebelas, dan jam kedua belas, dibayar 4 (empat) kali Upah sejam. Pasal 32 (1) Perhitun



## 12. HyDE - Hypothetical Document Embeddings

In [21]:
def generate_hypothetical_answers(question, n=2, max_new_tokens=150):
    hypothetical_prompt = (
        f"Jawab pertanyaan hukum ketenagakerjaan berikut secara singkat berdasarkan "
        f"perkiraan Anda, walau Anda tidak yakin dan belum melihat dokumen resminya:\n"
        f"{question}"
    )
    answers = []
    for _ in range(n):
        ans = generate_answer(hypothetical_prompt, max_new_tokens=max_new_tokens)
        answers.append(ans)
    return answers


def hyde_query_vector(question, n_hypothetical=2):
    hypothetical_answers = generate_hypothetical_answers(question, n=n_hypothetical)

    query_vec = np.array(embeddings.embed_query(question))
    hyp_vecs = [np.array(v) for v in embeddings.embed_documents(hypothetical_answers)]

    combined = np.mean([query_vec] + hyp_vecs, axis=0)
    combined = combined / np.linalg.norm(combined)
    return combined.tolist(), hypothetical_answers


In [22]:
demo_vector, demo_hyde_answers = hyde_query_vector(
    "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"
)
print(f"Jumlah jawaban halusinasi dibuat: {len(demo_hyde_answers)}")
for i, ans in enumerate(demo_hyde_answers, 1):
    print(f"\nHalusinasi #{i}:", ans[:200])


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Jumlah jawaban halusinasi dibuat: 2

Halusinasi #1: assistant
Tentu saja, Anda berhak mendapatkan uang lembur. Sebagai staf admin, Anda memiliki hak untuk mendapatkan bayaran tambahan untuk waktu yang dihabiskan bekerja di luar jam kerja rutin. Namun, 

Halusinasi #2: assistant
Tidak mungkin saya memberikan jawaban yang akurat tanpa melihat dokumen resmi seperti kontrak kerja atau peraturan kantor Anda, tetapi dalam banyak kasus, staf admin dapat berhak mendapatkan


## 13. Reranker (Cross-Encoder) + Top-K

In [23]:
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)

def rerank_documents(query, docs, top_k=3):
    if not docs:
        return []
    pairs = [(query, d.page_content) for d in docs]
    raw_scores = reranker.predict(pairs)
    scored = list(zip(docs, raw_scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

## 14. Threshold Relevansi + Fallback DuckDuckGo Search

In [24]:
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

RELEVANCE_THRESHOLD = 0.5

def duckduckgo_search_fallback(question, max_results=3):
    with DDGS() as ddgs:
        results = list(ddgs.text(question, max_results=max_results, region="id-id"))
    context_parts = []
    citations = []
    for i, r in enumerate(results, 1):
        snippet = r.get("body", "")
        url = r.get("href", "")
        context_parts.append(f"[{i}] {snippet}")
        citations.append(f"[{i}] {url}")
    context = "\n\n".join(context_parts)
    return context, citations


## 15. Pipeline RAG Advanced Lengkap

In [35]:
def legal_rag_pipeline_advanced(question):
    hyde_vector, hyde_answers = hyde_query_vector(question, n_hypothetical=2)

    semantic_docs = faiss_vectorstore.similarity_search_by_vector(hyde_vector, k=5)
    bm25_docs = bm25_retriever.invoke(question)[:5]

    combined_docs, seen = [], set()
    for d in semantic_docs + bm25_docs:
        key = d.page_content[:200]
        if key not in seen:
            seen.add(key)
            combined_docs.append(d)

    reranked = rerank_documents(question, combined_docs, top_k=3)

    if reranked:
        top1_score = sigmoid(float(reranked[0][1]))
    else:
        top1_score = 0.0

    if not reranked or top1_score < RELEVANCE_THRESHOLD:
        source_used = "duckduckgo"
        context, citations = duckduckgo_search_fallback(question)
    else:
        source_used = "dokumen_lokal"
        context, citations = format_context_with_citations([d for d, _ in reranked])

    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    answer = generate_answer(prompt)

    return {
        "answer": answer,
        "source_used": source_used,
        "top1_relevance_score": top1_score,
        "citations": citations,
        "hyde_hypothetical_answers": hyde_answers,
    }

In [38]:
test_case_question = (
    "Saya adalah pemilik usaha keci, yang baru saja memperkejakan 3 pegawai. "
    "Apakah ada syarat masa percobaan (probation) untuk pekerja PKWT berdasarkan ketentuan regulasi ini?"
)

result = legal_rag_pipeline_advanced(test_case_question)

print("Sumber yang dipakai :", result["source_used"])
print("Skor relevansi Top-1:", round(result["top1_relevance_score"], 3))
print("\nJawaban:\n", result["answer"])
print("\nSitasi:")
for c in result["citations"]:
    print(" -", c)

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Sumber yang dipakai : dokumen_lokal
Skor relevansi Top-1: 0.714

Jawaban:
 Berdasarkan ketentuan pasal 12 dan 13 Peraturan Perusahaan Waktu Tertentu (PKWT) yang diterbitkan oleh Presiden Republik Indonesia, masa percobaan kerja (probation) tidak dapat disyaratkan untuk pekerja PKWT. Masa percobaan kerja yang disyaratkan akan batal demi hukum dan masa kerja tetap dihitung. Namun, detail lebih lanjut tentang masa percobaan kerja yang disyaratkan harus ditentukan dalam PKWT.

Sitasi:
 - [1] PP No. 35/2021, hal. 9
 - [2] PP No. 35/2021, hal. 6
 - [3] UU No. 6/2023, hal. 553


In [37]:
def rag_interface_fn(question):
    result = legal_rag_pipeline_advanced(question)
    citation_text = "\n".join(result["citations"])
    return (
        f"{result['answer']}\n\n"
        f"---\n"
        f"Sumber: {result['source_used']} | skor relevansi top-1: {result['top1_relevance_score']:.3f}\n"
        f"{citation_text}"
    )

advanced_demo = gr.Interface(
    fn=rag_interface_fn,
    inputs=gr.Textbox(label="Pertanyaan"),
    outputs=gr.Textbox(label="Jawaban + Sitasi"),
    title="Chatbot Tim Legal",
)
advanced_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e91e66b32ce7f1ef0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
